# 02 — Study areas, spatial splits, and GEDI support

This notebook reconstructs the three study-area panels, spatially disjoint TRAIN/VAL/TEST partitions, and GEDI RH95 height-class counts used in Figs. 1–2 and Table 1.

In [ ]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.patheffects as path_effects
from matplotlib.patches import Patch
from matplotlib.colors import Normalize, LinearSegmentedColormap

import rasterio
from rasterio.features import shapes
from rasterio.warp import reproject, Resampling
from shapely.geometry import box, shape
from shapely.ops import unary_union
import geopandas as gpd

try:
    import contextily as cx
    HAS_CONTEXTILY = True
except Exception as exc:
    HAS_CONTEXTILY = False
    print("[INFO] contextily unavailable; local Sentinel-2 RGB fallback will be used:", exc)

PUBLICATION_ROOT = Path(r"C:\Users\Dell\Desktop\Publication_Clarck\Natural_Sampling")
# Inputs remain shared with the verified official pipeline; only outputs are isolated.
SOURCE_DATA_ROOT = PUBLICATION_ROOT.parent
OUT_DIR = PUBLICATION_ROOT / "Results" / "Final_Article" / "Study_Area"
OUT_DIR.mkdir(parents=True, exist_ok=True)

PATCH_SIZE = 512
MAP_CONTEXT_PADDING = 0.10  # 10% context around the raster/AOI
BACKGROUND_MODE = "dynamic_world_preferred"
SHOW_AOI_OUTLINE = True
EXPORT_DPI = 600

# ESA WorldCover 2021 binary tree-cover masks used to derive a continuous
# local percentage. Agadir is stored under the historical Taroudant AOI name.
TREE_COVER_MASK_RASTERS = {
    "Ifran": Path(r"E:\CHM\_ESA_WORLDCOVER_MASKS\Ifran_6\ESA_CLASS10_TREE_COVER_MASK_MATCH_REF.tif"),
    "Maamoura": Path(r"E:\CHM\_ESA_WORLDCOVER_MASKS\Maamoura\ESA_CLASS10_TREE_COVER_MASK_MATCH_REF.tif"),
    "Agadir": Path(r"E:\CHM\_ESA_WORLDCOVER_MASKS\Taroudant\ESA_FOREST_TREE10_MASK_MATCH_REF.tif"),
}
TREE_DENSITY_WINDOW_M = 250.0

# Preferred continuous backgrounds created by
# Dynamic_World_Tree_Density_Three_Forests.ipynb. The study-area notebook
# falls back to the verified ESA WorldCover density when an output is absent.
DYNAMIC_WORLD_DENSITY_RASTERS = {
    forest: SOURCE_DATA_ROOT / "Data" / "Study_Area" / "Dynamic_World_2020_10m" / forest
    / f"{forest}_DynamicWorld_2020_tree_density_proxy_250m_ALIGNED_10m.tif"
    for forest in ("Ifran", "Maamoura", "Agadir")
}

SITES = {
    "Ifran": {
        "ecosystem": "Moderately dense",
        "catalog": SOURCE_DATA_ROOT / "Data" / "Dense" / "Ifran" / "Catalogs" / "final_catalog_C15_NATIVE",
        "s2_dir": Path(r"E:\CHM\Ifran_6\DATA\S2\S2_MONTHLY_IFRAN_CLEAN_V1"),
    },
    "Maamoura": {
        "ecosystem": "Low density",
        "catalog": SOURCE_DATA_ROOT / "Data" / "Low_Sparsity" / "Maamoura" / "Catalogs" / "final_catalog",
        "s2_dir": Path(r"E:\CHM\Maamoura\Data\S2_12_Bands"),
    },
    "Agadir": {
        "ecosystem": "Sparse",
        "catalog": SOURCE_DATA_ROOT / "Data" / "Sparse" / "Agadir" / "Catalogs" / "final_catalog",
        "s2_dir": Path(r"E:\CHM\Agadir\DATA\S2_12_Bands"),
    },
}

SPLIT_STYLE = {
    "train": {"facecolor": "#7AAED1", "edgecolor": "#397FA8", "alpha": 0.52},
    "val":   {"facecolor": "#E77B70", "edgecolor": "#B94E45", "alpha": 0.62},
    "test":  {"facecolor": "#56A96B", "edgecolor": "#23783A", "alpha": 0.62},
}
SPLIT_LABEL = {"train": "Train", "val": "Val", "test": "Test"}

mpl.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 9,
    "axes.titlesize": 11,
    "axes.labelsize": 9,
    "legend.fontsize": 8,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
})

print("Output:", OUT_DIR)
print("Background mode:", BACKGROUND_MODE)


In [ ]:
def first_s2_raster(directory: Path) -> Path:
    candidates = sorted(directory.glob("*.tif"))
    if not candidates:
        raise FileNotFoundError(f"No Sentinel-2 GeoTIFF found in {directory}")
    return candidates[0]


def load_site(name: str, cfg: dict) -> dict:
    catalog = cfg["catalog"]
    sample_path = catalog / "sample_catalog.csv"
    manifest_path = catalog / "spatial_split_manifest.csv"
    for path in (sample_path, manifest_path):
        if not path.is_file():
            raise FileNotFoundError(path)

    samples = pd.read_csv(sample_path)
    manifest = pd.read_csv(manifest_path)
    required = {"patch_key", "patch_row_start", "patch_col_start"}
    missing = required - set(samples.columns)
    if missing:
        raise RuntimeError(f"{name}: missing sample columns {sorted(missing)}")
    if not {"patch_key", "split"}.issubset(manifest.columns):
        raise RuntimeError(f"{name}: invalid spatial_split_manifest.csv")

    patches = (
        samples[["patch_key", "patch_row_start", "patch_col_start"]]
        .drop_duplicates("patch_key")
        .merge(manifest[["patch_key", "split"]], on="patch_key", how="left", validate="one_to_one")
    )
    patches["split"] = patches["split"].replace({"validation": "val", "valid": "val"})
    if patches["split"].isna().any():
        raise RuntimeError(f"{name}: some patches have no frozen split assignment")
    unknown = set(patches["split"]) - set(SPLIT_STYLE)
    if unknown:
        raise RuntimeError(f"{name}: unknown split labels {sorted(unknown)}")

    reference = first_s2_raster(cfg["s2_dir"])
    with rasterio.open(reference) as src:
        if src.crs is None:
            raise RuntimeError(f"{name}: Sentinel-2 reference has no CRS")
        profile = {
            "crs": src.crs,
            "transform": src.transform,
            "width": src.width,
            "height": src.height,
            "bounds": src.bounds,
            "count": src.count,
            "res": src.res,
            "nodata": src.nodata,
        }

    return {
        **cfg,
        "name": name,
        "samples": samples,
        "patches": patches,
        "reference": reference,
        "profile": profile,
    }


SITE_DATA = {name: load_site(name, cfg) for name, cfg in SITES.items()}

preflight_rows = []
for name, data in SITE_DATA.items():
    counts = data["patches"]["split"].value_counts()
    p = data["profile"]
    preflight_rows.append({
        "forest": name,
        "catalogue": str(data["catalog"]),
        "reference_raster": str(data["reference"]),
        "CRS": p["crs"].to_string(),
        "pixel_size_m": abs(float(p["transform"].a)),
        "train_patches": int(counts.get("train", 0)),
        "val_patches": int(counts.get("val", 0)),
        "test_patches": int(counts.get("test", 0)),
    })

preflight = pd.DataFrame(preflight_rows)
display(preflight)
preflight.to_csv(OUT_DIR / "Study_Area_preflight.csv", index=False)

assert set(preflight["CRS"]) == {"EPSG:32629", "EPSG:32630"}, preflight
assert np.allclose(preflight["pixel_size_m"], 10.0), preflight
print("[PASS] Final split catalogues and Sentinel-2 map grids are valid.")


## Geospatial reconstruction and publication maps

All layers are checked against their recorded CRS and raster grid before plotting.

In [ ]:
def patch_geodataframe(data: dict) -> gpd.GeoDataFrame:
    tr = data["profile"]["transform"]
    width = data["profile"]["width"]
    height = data["profile"]["height"]
    records = []
    for row in data["patches"].itertuples(index=False):
        r0 = max(0, int(row.patch_row_start))
        c0 = max(0, int(row.patch_col_start))
        r1 = min(height, r0 + PATCH_SIZE)
        c1 = min(width, c0 + PATCH_SIZE)
        if r1 <= r0 or c1 <= c0:
            continue
        x0, y0 = tr * (c0, r0)
        x1, y1 = tr * (c1, r1)
        records.append({
            "patch_key": row.patch_key,
            "split": row.split,
            "geometry": box(min(x0, x1), min(y0, y1), max(x0, x1), max(y0, y1)),
        })
    return gpd.GeoDataFrame(records, geometry="geometry", crs=data["profile"]["crs"])


def valid_aoi_geometry(data: dict):
    with rasterio.open(data["reference"]) as src:
        valid = src.dataset_mask() > 0
        geoms = [
            shape(geom)
            for geom, value in shapes(valid.astype(np.uint8), mask=valid, transform=src.transform)
            if value == 1
        ]
    return unary_union(geoms) if geoms else None


def robust_rgb(data: dict) -> tuple[np.ndarray, tuple]:
    with rasterio.open(data["reference"]) as src:
        # Raw source order is B02, B03, B04, ... -> display R,G,B = 3,2,1.
        rgb = src.read([3, 2, 1]).astype(np.float32)
        extent = (src.bounds.left, src.bounds.right, src.bounds.bottom, src.bounds.top)
        invalid = src.dataset_mask() == 0
    out = np.zeros_like(rgb, dtype=np.float32)
    for i in range(3):
        band = rgb[i]
        finite = np.isfinite(band) & ~invalid
        if finite.any():
            lo, hi = np.nanpercentile(band[finite], [2, 98])
            out[i] = np.clip((band - lo) / max(hi - lo, 1e-6), 0, 1)
    out[:, invalid] = np.nan
    return np.moveaxis(out, 0, -1), extent


def load_density_on_reference(data: dict, density_path: Path) -> np.ndarray:
    p = data["profile"]
    destination = np.full((p["height"], p["width"]), np.nan, dtype=np.float32)
    with rasterio.open(density_path) as src:
        source = src.read(1, masked=True).filled(np.nan).astype(np.float32)
        reproject(
            source=source,
            destination=destination,
            src_transform=src.transform,
            src_crs=src.crs,
            dst_transform=p["transform"],
            dst_crs=p["crs"],
            src_nodata=np.nan,
            dst_nodata=np.nan,
            resampling=Resampling.bilinear,
        )
    finite = destination[np.isfinite(destination)]
    if finite.size == 0:
        raise RuntimeError(f"Density raster does not overlap {data['name']}: {density_path}")
    if finite.min() < -1e-3 or finite.max() > 100.001:
        raise RuntimeError(f"Density raster must be 0–100%, found {finite.min()}–{finite.max()}")
    return destination


def _box_sum(array: np.ndarray, window: int) -> np.ndarray:
    """Centred moving-window sum with constant padding and unchanged shape."""
    before = window // 2
    after = window - 1 - before
    padded = np.pad(array, ((before, after), (before, after)), mode="constant")
    integral = np.pad(padded, ((1, 0), (1, 0)), mode="constant")
    integral = integral.cumsum(axis=0, dtype=np.float64).cumsum(axis=1, dtype=np.float64)
    return (
        integral[window:, window:]
        - integral[:-window, window:]
        - integral[window:, :-window]
        + integral[:-window, :-window]
    )


def load_worldcover_tree_density(data: dict, mask_path: Path) -> tuple[np.ndarray, dict]:
    """Reproject a binary ESA tree mask and derive local percent cover."""
    if not mask_path.is_file():
        raise FileNotFoundError(mask_path)
    p = data["profile"]
    tree = np.zeros((p["height"], p["width"]), dtype=np.uint8)
    valid = np.zeros((p["height"], p["width"]), dtype=np.uint8)
    with rasterio.open(mask_path) as src:
        raw = src.read(1)
        # A matched binary mask may declare 0 as nodata although 0 is also
        # the scientifically meaningful non-tree class. In a pure 0/1 mask,
        # both classes are valid; external nodata codes such as 255 are excluded.
        finite_values = np.unique(raw[np.isfinite(raw)])
        is_binary_01 = set(finite_values.tolist()).issubset({0, 1})
        src_valid = np.ones(raw.shape, dtype=np.uint8)
        if src.nodata is not None and not (is_binary_01 and float(src.nodata) == 0.0):
            src_valid[raw == src.nodata] = 0
        src_tree = ((raw > 0) & (src_valid > 0)).astype(np.uint8)
        reproject(
            source=src_tree, destination=tree,
            src_transform=src.transform, src_crs=src.crs,
            dst_transform=p["transform"], dst_crs=p["crs"],
            src_nodata=0, dst_nodata=0, resampling=Resampling.nearest,
        )
        reproject(
            source=src_valid, destination=valid,
            src_transform=src.transform, src_crs=src.crs,
            dst_transform=p["transform"], dst_crs=p["crs"],
            src_nodata=0, dst_nodata=0, resampling=Resampling.nearest,
        )

    pixel_m = abs(float(p["transform"].a))
    window_px = max(3, int(round(TREE_DENSITY_WINDOW_M / pixel_m)))
    if window_px % 2 == 0:
        window_px += 1
    numerator = _box_sum(tree.astype(np.float32), window_px)
    denominator = _box_sum(valid.astype(np.float32), window_px)
    density = np.full(tree.shape, np.nan, dtype=np.float32)
    np.divide(100.0 * numerator, denominator, out=density, where=denominator > 0)
    if not np.isfinite(density).any():
        raise RuntimeError(f"{data['name']}: ESA tree mask has no overlap with the reference grid")
    provenance = {
        "forest": data["name"],
        "source": str(mask_path),
        "source_product": "ESA WorldCover 2021 v200 tree-cover class",
        "derivation": "binary tree class -> centred moving-window percent cover",
        "window_m": TREE_DENSITY_WINDOW_M,
        "window_pixels": window_px,
        "destination_crs": p["crs"].to_string(),
        "destination_pixel_m": pixel_m,
        "finite_fraction": float(np.isfinite(density).mean()),
        "density_min_pct": float(np.nanmin(density)),
        "density_max_pct": float(np.nanmax(density)),
    }
    return density, provenance


for data in SITE_DATA.values():
    data["patch_gdf"] = patch_geodataframe(data)
    data["aoi_geometry"] = valid_aoi_geometry(data)


def choose_background(data: dict) -> str:
    if BACKGROUND_MODE == "dynamic_world_preferred":
        dynamic_path = DYNAMIC_WORLD_DENSITY_RASTERS[data["name"]]
        if dynamic_path.is_file():
            data["density"] = load_density_on_reference(data, dynamic_path)
            data["density_provenance"] = {
                "forest": data["name"],
                "source": str(dynamic_path),
                "source_product": "Dynamic World 2020 mean trees probability",
                "derivation": "annual trees probability -> 250 m local mean; exact 10 m grid",
                "window_m": TREE_DENSITY_WINDOW_M,
                "window_pixels": int(round(TREE_DENSITY_WINDOW_M / 10.0)),
                "destination_crs": data["profile"]["crs"].to_string(),
                "destination_pixel_m": abs(float(data["profile"]["transform"].a)),
                "finite_fraction": float(np.isfinite(data["density"]).mean()),
                "density_min_pct": float(np.nanmin(data["density"])),
                "density_max_pct": float(np.nanmax(data["density"])),
            }
            return "tree_cover_density"
        warnings.warn(
            f"{data['name']}: Dynamic World density is absent; using verified ESA WorldCover fallback. "
            "Execute Dynamic_World_Tree_Density_Three_Forests.ipynb to activate it."
        )
    elif BACKGROUND_MODE != "esa_worldcover_density":
        raise ValueError(f"Unsupported BACKGROUND_MODE={BACKGROUND_MODE!r}")

    esa_path = TREE_COVER_MASK_RASTERS[data["name"]]
    data["density"], data["density_provenance"] = load_worldcover_tree_density(data, esa_path)
    return "tree_cover_density"


for data in SITE_DATA.values():
    data["background"] = choose_background(data)

background_report = pd.DataFrame([
    {
        "forest": name,
        "background_used": data["background"],
        "density_path": data["density_provenance"]["source"],
        "density_source_product": data["density_provenance"]["source_product"],
        "density_window_m": TREE_DENSITY_WINDOW_M,
        "CRS": data["profile"]["crs"].to_string(),
    }
    for name, data in SITE_DATA.items()
])
display(background_report)
background_report.to_csv(OUT_DIR / "Study_Area_background_provenance.csv", index=False)


density_provenance = pd.DataFrame([
    data["density_provenance"] for data in SITE_DATA.values()
])
display(density_provenance)
density_provenance.to_csv(
    OUT_DIR / "ESA_WorldCover_tree_density_provenance.csv", index=False
)


In [ ]:
# STUDY_AREA_MAP_LAYOUT_V2
def contextual_map_bounds(data, raster_bounds, padding_fraction=MAP_CONTEXT_PADDING):
    # Pad the AOI while remaining inside the raster's valid spatial coverage.
    focus = data["aoi_geometry"].bounds if data["aoi_geometry"] is not None else raster_bounds
    xmin, ymin, xmax, ymax = focus
    width = xmax - xmin
    height = ymax - ymin
    pad_x = width * padding_fraction
    pad_y = height * padding_fraction
    rxmin, rymin, rxmax, rymax = raster_bounds
    return (
        max(rxmin, xmin - pad_x), max(rymin, ymin - pad_y),
        min(rxmax, xmax + pad_x), min(rymax, ymax + pad_y),
    )


def add_north_arrow(ax):
    ax.annotate(
        "N", xy=(0.065, 0.965), xytext=(0.065, 0.825),
        xycoords="axes fraction", textcoords="axes fraction",
        ha="center", va="center", fontsize=9, fontweight="bold",
        arrowprops=dict(facecolor="black", edgecolor="black", width=2.0, headwidth=7.5),
        zorder=30,
    )


def nice_scale_length(width_m: float) -> float:
    target = width_m * 0.22
    candidates = np.array([100, 200, 500, 1000, 2000, 5000, 10000, 20000], dtype=float)
    return float(candidates[np.argmin(np.abs(candidates - target))])


def add_scale_bar(ax, bounds):
    xmin, ymin, xmax, ymax = bounds
    width = xmax - xmin
    height = ymax - ymin
    length = nice_scale_length(width)
    x0 = xmax - 0.07 * width - length
    y0 = ymin + 0.065 * height
    ax.plot([x0, x0 + length], [y0, y0], color="white", lw=4.0, solid_capstyle="butt", zorder=31)
    ax.plot([x0, x0 + length], [y0, y0], color="black", lw=0.9, solid_capstyle="butt", zorder=32)
    for fraction in (0, 0.5, 1):
        x = x0 + fraction * length
        ax.plot([x, x], [y0 - 0.012 * height, y0 + 0.012 * height],
                color="white", lw=2.2, zorder=31)
        ax.plot([x, x], [y0 - 0.012 * height, y0 + 0.012 * height],
                color="black", lw=0.6, zorder=32)
    labels = ["0", f"{length/2000:g}", f"{length/1000:g} km"] if length >= 1000 else ["0", f"{length/2:g}", f"{length:g} m"]
    for fraction, label in zip((0, 0.5, 1), labels):
        ax.text(x0 + fraction * length, y0 + 0.025 * height, label,
                ha="center", va="bottom", fontsize=6.8, color="white", fontweight="semibold",
                path_effects=[path_effects.withStroke(linewidth=1.4, foreground="black")],
                zorder=33)


def display_background(ax, data: dict) -> tuple[str, object | None]:
    p = data["profile"]
    bounds = (p["bounds"].left, p["bounds"].bottom, p["bounds"].right, p["bounds"].top)
    mode = data["background"]
    mappable = None
    if mode == "tree_cover_density":
        density_cmap = LinearSegmentedColormap.from_list(
            "tree_density", ["#ffffff", "#ffffb2", "#7fc97f", "#006d2c"]
        )
        mappable = ax.imshow(
            data["density"], extent=(bounds[0], bounds[2], bounds[1], bounds[3]),
            origin="upper", cmap=density_cmap, vmin=0, vmax=100, interpolation="nearest",
            zorder=0,
        )
    elif mode == "esri":
        ax.set_xlim(bounds[0], bounds[2])
        ax.set_ylim(bounds[1], bounds[3])
        try:
            cx.add_basemap(
                ax, crs=p["crs"], source=cx.providers.Esri.WorldImagery,
                attribution=False, reset_extent=True, zoom="auto",
            )
        except Exception as exc:
            warnings.warn(f"{data['name']}: ESRI failed ({exc}); using Sentinel-2 RGB")
            rgb, extent = robust_rgb(data)
            ax.imshow(rgb, extent=extent, origin="upper", interpolation="bilinear", zorder=0)
            mode = "sentinel2_fallback"
    else:
        rgb, extent = robust_rgb(data)
        ax.imshow(rgb, extent=extent, origin="upper", interpolation="bilinear", zorder=0)
    return mode, mappable


def plot_site_panel(ax, data: dict, letter: str | None = None):
    p = data["profile"]
    raster_bounds = (
        p["bounds"].left, p["bounds"].bottom,
        p["bounds"].right, p["bounds"].top,
    )
    bounds = contextual_map_bounds(data, raster_bounds)
    mode, mappable = display_background(ax, data)

    if SHOW_AOI_OUTLINE and data["aoi_geometry"] is not None:
        gpd.GeoSeries([data["aoi_geometry"]], crs=p["crs"]).boundary.plot(
            ax=ax, color="black", linewidth=1.15, zorder=12
        )

    for split in ("train", "val", "test"):
        part = data["patch_gdf"][data["patch_gdf"]["split"].eq(split)]
        if len(part):
            style = SPLIT_STYLE[split]
            part.plot(
                ax=ax, facecolor=style["facecolor"], edgecolor=style["edgecolor"],
                alpha=style["alpha"], linewidth=1.05, zorder=15,
            )

    ax.set_xlim(bounds[0], bounds[2])
    ax.set_ylim(bounds[1], bounds[3])
    ax.set_aspect("equal", adjustable="box")
    ax.set_axis_off()
    ax.set_title(f"{data['ecosystem']} — {data['name']} Forest", pad=5, fontweight="semibold")
    if letter:
        ax.text(0.985, 0.985, f"({letter})", transform=ax.transAxes, ha="right", va="top",
                fontsize=9, fontweight="bold", color="black",
                bbox=dict(facecolor="white", edgecolor="none", alpha=0.78, pad=1.2), zorder=40)
    add_north_arrow(ax)
    add_scale_bar(ax, bounds)
    return mode, mappable


def save_figure(fig, stem: str):
    paths = {
        "png": OUT_DIR / f"{stem}.png",
        "svg": OUT_DIR / f"{stem}.svg",
        "pdf": OUT_DIR / f"{stem}.pdf",
    }
    fig.savefig(paths["png"], dpi=EXPORT_DPI, bbox_inches="tight", facecolor="white")
    fig.savefig(paths["svg"], bbox_inches="tight", facecolor="white")
    fig.savefig(paths["pdf"], bbox_inches="tight", facecolor="white")
    return paths


fig, axes = plt.subplots(1, 3, figsize=(15.0, 7.8), constrained_layout=False)
fig.subplots_adjust(left=0.015, right=0.900, bottom=0.13, top=0.92, wspace=0.04)
used_backgrounds = []
density_mappable = None
for letter, (ax, (name, data)) in zip("abc", zip(axes, SITE_DATA.items())):
    mode, mappable = plot_site_panel(ax, data, letter)
    used_backgrounds.append({"forest": name, "background_rendered": mode})
    if mappable is not None:
        density_mappable = mappable

legend_handles = [
    Patch(facecolor=SPLIT_STYLE[s]["facecolor"], edgecolor=SPLIT_STYLE[s]["edgecolor"],
          alpha=SPLIT_STYLE[s]["alpha"], label=SPLIT_LABEL[s])
    for s in ("train", "val", "test")
]
legend_handles.append(Patch(facecolor="none", edgecolor="black", linewidth=1.15, label="Valid AOI boundary"))
fig.legend(handles=legend_handles, loc="lower center", ncol=4, frameon=True,
           bbox_to_anchor=(0.5, -0.015), columnspacing=1.8)

if density_mappable is not None:
    cax = fig.add_axes([0.920, 0.18, 0.014, 0.64])
    cbar = fig.colorbar(density_mappable, cax=cax)
    cbar.set_label("Tree-cover occupancy (%)")
    cbar.set_ticks([0, 25, 50, 75, 100])

combined_paths = save_figure(fig, "Fig_Study_Area_Three_Forests_Spatial_Splits")
plt.show()
plt.close(fig)

render_report = pd.DataFrame(used_backgrounds)
display(render_report)
render_report.to_csv(OUT_DIR / "Study_Area_rendered_backgrounds.csv", index=False)
print("\n".join(f"Saved {kind.upper()}: {path}" for kind, path in combined_paths.items()))


In [ ]:
individual_paths = {}
for name, data in SITE_DATA.items():
    fig, ax = plt.subplots(figsize=(6.4, 6.1), constrained_layout=True)
    plot_site_panel(ax, data)
    ax.legend(handles=[
        Patch(facecolor=SPLIT_STYLE[s]["facecolor"], edgecolor=SPLIT_STYLE[s]["edgecolor"],
              alpha=SPLIT_STYLE[s]["alpha"], label=SPLIT_LABEL[s])
        for s in ("train", "val", "test")
    ] + [Patch(facecolor="none", edgecolor="black", label="Valid AOI boundary")],
        loc="upper right", frameon=True)
    individual_paths[name] = save_figure(fig, f"Fig_Study_Area_{name}_Spatial_Split")
    plt.show()
    plt.close(fig)

for forest, paths in individual_paths.items():
    print(f"\n{forest}")
    print("\n".join(f"  {kind.upper()}: {path}" for kind, path in paths.items()))


## GEDI distributions and split diagnostics

The following cells reproduce the height-class composition and the combined study-area figure.

In [ ]:
GEDI_BINS = {
    "Ifran": np.arange(0.0, 45.0 + 5.0, 5.0),
    "Maamoura": np.arange(0.0, 20.0 + 5.0, 5.0),
    "Agadir": np.arange(0.0, 20.0 + 5.0, 5.0),
}
SPLIT_COLORS = {"train": "#7AAED1", "val": "#E77B70", "test": "#56A96B"}


def load_unique_gedi_shots(data: dict) -> pd.DataFrame:
    path = data["catalog"] / "shot_catalog_step05.csv.gz"
    if not path.is_file():
        raise FileNotFoundError(path)
    shots = pd.read_csv(path, usecols=["split", "rh95", "aux_shot_uid"])
    shots["split"] = shots["split"].replace({"validation": "val", "valid": "val"})
    shots["rh95"] = pd.to_numeric(shots["rh95"], errors="coerce")
    shots = shots[
        shots["split"].isin(SPLIT_COLORS)
        & np.isfinite(shots["rh95"])
    ].copy()
    conflicts = shots.groupby("aux_shot_uid", sort=False)["split"].nunique()
    if (conflicts > 1).any():
        raise RuntimeError(
            f"{data['name']}: a physical GEDI shot occurs in multiple frozen splits"
        )
    return shots.drop_duplicates("aux_shot_uid").reset_index(drop=True)


def binned_split_table(forest: str, shots: pd.DataFrame) -> pd.DataFrame:
    edges = GEDI_BINS[forest]
    labels = [f"{int(lo)}–{int(hi)} m" for lo, hi in zip(edges[:-1], edges[1:])]
    work = shots.copy()
    work["height_class"] = pd.cut(
        work["rh95"], bins=edges, labels=labels,
        include_lowest=True, right=False,
    )
    rows = []
    for split in ("train", "val", "test"):
        part = work[work["split"].eq(split)]
        counts = part["height_class"].value_counts(sort=False).reindex(labels, fill_value=0)
        total = int(len(part))
        for label, count in counts.items():
            rows.append({
                "forest": forest, "split": split, "height_class": str(label),
                "count": int(count), "split_n": total,
                "percent": 100.0 * int(count) / total if total else np.nan,
            })
    return pd.DataFrame(rows)


GEDI_SHOTS = {
    forest: load_unique_gedi_shots(data)
    for forest, data in SITE_DATA.items()
}
GEDI_DISTRIBUTIONS = pd.concat(
    [binned_split_table(forest, GEDI_SHOTS[forest]) for forest in SITE_DATA],
    ignore_index=True,
)
display(
    GEDI_DISTRIBUTIONS.groupby(["forest", "split"], as_index=False)
    .agg(unique_shots=("split_n", "first"), binned_shots=("count", "sum"))
)
GEDI_DISTRIBUTIONS.to_csv(
    OUT_DIR / "GEDI_RH95_unique_shot_distribution_by_split.csv", index=False
)


def _plot_one_gedi_distribution(ax, forest, data, value_column, ylabel):
    table = GEDI_DISTRIBUTIONS[GEDI_DISTRIBUTIONS["forest"].eq(forest)]
    labels = table["height_class"].drop_duplicates().tolist()
    x = np.arange(len(labels), dtype=float)
    width = 0.25
    for offset, split in zip((-1, 0, 1), ("train", "val", "test")):
        part = table[table["split"].eq(split)].set_index("height_class").reindex(labels)
        n = int(part["split_n"].iloc[0])
        is_test = split == "test"
        ax.bar(
            x + offset * width, part[value_column].to_numpy(float),
            width=width,
            color="#F2F2F2" if is_test else SPLIT_COLORS[split],
            edgecolor="black" if is_test else "none",
            linewidth=1.0 if is_test else 0.0,
            hatch="////" if is_test else None,
            alpha=1.0 if is_test else 0.90,
            zorder=3 if is_test else 2,
            label=SPLIT_LABEL[split],
        )
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=25, ha="right")
    ax.set_title(f"{data['ecosystem']} — {forest} Forest", fontweight="semibold")
    ax.set_xlabel("GEDI RH95 height class")
    ax.set_ylabel(ylabel)
    ax.grid(axis="y", alpha=0.22)
    ax.set_axisbelow(True)
    ax.legend(loc="upper right", frameon=True, framealpha=0.95, handlelength=2.2, handleheight=1.15, borderpad=0.45)


def plot_gedi_distributions(value_column: str, ylabel: str, stem: str):
    # Overview retained for supplementary material.
    fig, axes = plt.subplots(1, 3, figsize=(14.2, 4.25), constrained_layout=True)
    for ax, (forest, data) in zip(axes, SITE_DATA.items()):
        _plot_one_gedi_distribution(ax, forest, data, value_column, ylabel)
    paths = save_figure(fig, stem)
    plt.show()
    plt.close(fig)

    # Separate figures are the primary article assets. Each forest keeps its
    # own y-scale because sample sizes differ by more than one order of magnitude.
    separate = {}
    for forest, data in SITE_DATA.items():
        fig, ax = plt.subplots(figsize=(7.2, 4.8), constrained_layout=True)
        _plot_one_gedi_distribution(ax, forest, data, value_column, ylabel)
        forest_stem = f"{stem}_{forest}_SEPARATE"
        separate[forest] = save_figure(fig, forest_stem)
        plt.show()
        plt.close(fig)
    print("\n".join(f"Saved {kind.upper()}: {path}" for kind, path in paths.items()))
    for forest, forest_paths in separate.items():
        print(f"\n{forest}")
        print("\n".join(f"  {kind.upper()}: {path}" for kind, path in forest_paths.items()))
    return paths, separate


gedi_count_paths, gedi_count_separate_paths = plot_gedi_distributions(
    "count", "GEDI shots",
    "Fig_GEDI_RH95_Distribution_By_Split_Unique_Counts",
)
gedi_percent_paths, gedi_percent_separate_paths = plot_gedi_distributions(
    "percent", "Within-split observations (%)",
    "Fig_GEDI_RH95_Distribution_By_Split_Percent",
)


In [ ]:
# STUDY_AREA_SPLIT_GEDI_ARTICLE_V2_QGIS
from matplotlib.patches import Patch
from matplotlib.colorbar import ColorbarBase
from matplotlib.colors import LinearSegmentedColormap, Normalize
from PIL import Image
import shutil
import subprocess

ARTICLE_IMAGES_DIR = (
    PUBLICATION_ROOT / "Writing_Article" / "1_Article" / "1_Article" / "Images"
)
QGIS_SPLIT_PDFS = {
    "Ifran": ARTICLE_IMAGES_DIR / "Ifran_Split_Spatial.pdf",
    "Maamoura": ARTICLE_IMAGES_DIR / "Maamoura_Split_Spatial.pdf",
    "Agadir": ARTICLE_IMAGES_DIR / "Agadir_Split_Spatial.pdf",
}
QGIS_SPLIT_COLORS = {
    "train": "#7AAED1",
    "val": "#E77B70",
    "test": "#56A96B",
}
TREE_DENSITY_CMAP = LinearSegmentedColormap.from_list(
    "tree_density_martin_style", ["#F3FAF5", "#54B948", "#006400"]
)


def load_cropped_qgis_pdf(pdf_path, dpi=600, white_threshold=250, pad_px=10):
    pdf_path = Path(pdf_path)
    if not pdf_path.is_file():
        raise FileNotFoundError(f"QGIS split map is missing: {pdf_path}")
    renderer = Path(r"C:\Users\Dell\AppData\Local\Programs\MiKTeX\miktex\bin\x64\pdftoppm.exe")
    if not renderer.is_file():
        renderer = Path(shutil.which("pdftoppm") or "")
    if not renderer.is_file():
        raise FileNotFoundError("pdftoppm is required to render the QGIS PDFs")
    cache_dir = OUT_DIR / "_qgis_pdf_cache"
    cache_dir.mkdir(parents=True, exist_ok=True)
    png_path = cache_dir / f"{pdf_path.stem}_{dpi}dpi.png"
    subprocess.run(
        [str(renderer), "-f", "1", "-l", "1", "-singlefile", "-png", "-r", str(dpi),
         str(pdf_path), str(png_path.with_suffix(""))],
        check=True, capture_output=True, text=True,
    )
    rgb = np.asarray(Image.open(png_path).convert("RGB"))
    content = np.any(rgb < white_threshold, axis=2)
    rows, cols = np.where(content)
    if rows.size == 0:
        raise RuntimeError(f"No cartographic content detected in {pdf_path}")
    y0 = max(0, int(rows.min()) - pad_px)
    y1 = min(rgb.shape[0], int(rows.max()) + pad_px + 1)
    x0 = max(0, int(cols.min()) - pad_px)
    x1 = min(rgb.shape[1], int(cols.max()) + pad_px + 1)
    return rgb[y0:y1, x0:x1]


# QGIS PDFs are retained as reference assets only. The publication maps
# below are rendered natively to guarantee exact patch alignment.
for forest, data in SITE_DATA.items():
    present_splits = set(data["patch_gdf"]["split"].dropna().astype(str))
    if present_splits != {"train", "val", "test"}:
        raise RuntimeError(
            f"{forest}: expected frozen train/val/test patches, found {sorted(present_splits)}"
        )
    table = GEDI_DISTRIBUTIONS[GEDI_DISTRIBUTIONS["forest"].eq(forest)]
    for split in ("train", "val", "test"):
        part = table[table["split"].eq(split)]
        if part.empty:
            raise RuntimeError(f"{forest}: missing GEDI distribution for split={split}")
        binned_n = int(part["count"].sum())
        split_n = int(part["split_n"].iloc[0])
        if binned_n <= 0 or binned_n > split_n:
            raise RuntimeError(
                f"{forest}/{split}: invalid binned GEDI count "
                f"({binned_n} displayed from {split_n} unique shots)."
            )


def add_split_legend(ax):
    handles = [
        Patch(
            facecolor="#F2F2F2" if split == "test" else QGIS_SPLIT_COLORS[split],
            edgecolor="black" if split == "test" else SPLIT_STYLE[split]["edgecolor"],
            linewidth=1.0 if split == "test" else 0.8,
            hatch="////" if split == "test" else None,
            alpha=1.0 if split == "test" else 0.82,
            label=SPLIT_LABEL[split],
        )
        for split in ("train", "val", "test")
    ]
    ax.legend(
        handles=handles,
        loc="upper right",
        frameon=True,
        framealpha=0.90,
        fontsize=8.0,
        borderpad=0.45,
        handlelength=2.2,
        handleheight=1.15,
        labelspacing=0.35,
    )


def add_compact_tree_density_key(ax):
    # Publication key outside the map, on its right-hand side.
    ax.text(
        1.086, 0.655, "Tree cover\noccupancy",
        transform=ax.transAxes, ha="center", va="bottom",
        fontsize=6.4, linespacing=1.10, clip_on=False,
        bbox=dict(facecolor="white", edgecolor="none", alpha=1.0, pad=0.6),
        zorder=50,
    )
    cax = ax.inset_axes([1.058, 0.315, 0.034, 0.255], zorder=50)
    cax.set_facecolor("white")
    cbar = ColorbarBase(
        cax, cmap=TREE_DENSITY_CMAP, norm=Normalize(vmin=1, vmax=100),
        orientation="vertical",
    )
    cbar.set_ticks([1, 100])
    cbar.set_ticklabels(["1%", "100%"])
    cbar.ax.yaxis.set_ticks_position("right")
    cbar.ax.tick_params(labelsize=7.0, length=2.0, width=0.45, pad=2.5)
    cbar.outline.set_linewidth(0.55)
    cax.set_clip_on(False)

def add_map_orientation_and_scale(ax, forest, data):
    # Compact north arrow at upper left, clear of the tree-density key.
    ax.annotate(
        "N", xy=(0.075, 0.940), xytext=(0.075, 0.800),
        xycoords="axes fraction", textcoords="axes fraction",
        ha="center", va="center", fontsize=9, fontweight="bold",
        arrowprops=dict(facecolor="black", edgecolor="white", linewidth=0.7,
                        width=2.0, headwidth=7.5, headlength=8.5),
        zorder=70,
    )
    # Derive a defensible metric length from the native projected raster width.
    p = data["profile"]
    width_m = float(p["bounds"].right - p["bounds"].left)
    length_m = nice_scale_length(width_m)
    frac = min(0.28, max(0.13, length_m / max(width_m, 1.0)))
    # Keep the full scale and its right-hand label safely inside the map.
    x1, x0, y = 0.965, 0.965 - frac, 0.052
    ax.plot([x0, x1], [y, y], transform=ax.transAxes, color="white", lw=4.2,
            solid_capstyle="butt", zorder=71)
    ax.plot([x0, x1], [y, y], transform=ax.transAxes, color="black", lw=1.0,
            solid_capstyle="butt", zorder=72)
    for x in (x0, (x0 + x1) / 2, x1):
        ax.plot([x, x], [y - 0.014, y + 0.014], transform=ax.transAxes,
                color="black", lw=0.8, zorder=72)
    labels = (["0", f"{length_m/2000:g}", f"{length_m/1000:g} km"]
              if length_m >= 1000 else ["0", f"{length_m/2:g}", f"{length_m:g} m"])
    for x, label in zip((x0, (x0 + x1) / 2, x1), labels):
        ax.text(x, y + 0.024, label, transform=ax.transAxes, ha="center", va="bottom",
                fontsize=6.8, color="black",
                bbox=dict(facecolor="white", edgecolor="none", alpha=0.78, pad=0.35),
                zorder=73)


def plot_split_and_gedi_pair(forest, data, axes, letters=None):
    hist_ax, map_ax = axes
    # Render background and patches in one native projected coordinate system.
    # This avoids the offset caused by assigning a geospatial extent to a
    # tightly cropped QGIS screenshot.
    p = data["profile"]
    raster_extent = (
        p["bounds"].left, p["bounds"].right,
        p["bounds"].bottom, p["bounds"].top,
    )
    map_ax.imshow(
        data["density"], extent=raster_extent, origin="upper",
        cmap=TREE_DENSITY_CMAP, vmin=1, vmax=100,
        aspect="equal", interpolation="nearest", zorder=0,
    )

    # Draw every split polygon once, with its fill and black cell boundary
    # produced from the same geometry. Adjacent patches therefore remain
    # individually visible and cannot be shifted relative to their fill.
    for split in ("train", "val", "test"):
        part = data["patch_gdf"][data["patch_gdf"]["split"].eq(split)]
        if len(part):
            is_test = split == "test"
            part.plot(
                ax=map_ax,
                facecolor="#F2F2F2" if is_test else QGIS_SPLIT_COLORS[split],
                edgecolor="black",
                linewidth=0.78,
                hatch="////" if is_test else None,
                alpha=1.0 if is_test else 0.58,
                zorder=20,
            )

    if data.get("aoi_geometry") is not None:
        gpd.GeoSeries([data["aoi_geometry"]], crs=p["crs"]).boundary.plot(
            ax=map_ax, color="black", linewidth=1.15, zorder=30,
        )

    # Slight zoom-out: keep AOI and outer patches away from the black frame.
    xmin, xmax, ymin, ymax = raster_extent
    center_x = 0.5 * (xmin + xmax)
    center_y = 0.5 * (ymin + ymax)
    square_span = 1.11 * max(xmax - xmin, ymax - ymin)
    half_span = 0.5 * square_span
    map_ax.set_xlim(center_x - half_span, center_x + half_span)
    map_ax.set_ylim(center_y - half_span, center_y + half_span)
    map_ax.set_facecolor("#F7FAF7")
    map_ax.set_anchor("W")
    map_ax.set_box_aspect(1.0)
    map_ax.set_xticks([])
    map_ax.set_yticks([])
    # Match the GEDI histogram with a clean black frame around the map.
    for spine in map_ax.spines.values():
        spine.set_visible(True)
        spine.set_color("black")
        spine.set_linewidth(0.9)
        spine.set_zorder(40)
    map_ax.set_title("")
    add_compact_tree_density_key(map_ax)
    add_map_orientation_and_scale(map_ax, forest, data)
    if letters is not None:
        map_ax.text(
            0.985, 0.018, f"({letters[1]})", transform=map_ax.transAxes,
            ha="right", va="bottom", fontsize=9, fontweight="bold",
            bbox=dict(facecolor="white", edgecolor="none", alpha=0.78, pad=1.2),
            zorder=60,
        )

    _plot_one_gedi_distribution(
        hist_ax,
        forest,
        data,
        value_column="count",
        ylabel="Unique GEDI RH95 shots",
    )
    if letters is not None:
        hist_ax.text(
            0.012, 0.985, f"({letters[0]})",
            transform=hist_ax.transAxes,
            ha="left", va="top",
            fontsize=9, fontweight="bold",
            bbox=dict(facecolor="white", edgecolor="none", alpha=0.78, pad=1.2),
            zorder=40,
        )
    hist_ax.set_title("")
    hist_ax.set_box_aspect(1.0)
    # With wspace=0, east/west anchoring makes the two black frames touch.
    hist_ax.set_anchor("E")
    map_ax.set_anchor("W")
    return map_ax


# One publication-ready overview: balanced panels and one centred title per row.
fig, axes = plt.subplots(
    len(SITE_DATA), 2,
    figsize=(12.0, 15.0),
    gridspec_kw={"width_ratios": [1.00, 1.00]},
    constrained_layout=False,
)
# Taller rows make the square maps larger; the reserved right margin holds
# the external tree-density keys without shrinking or clipping the maps.
fig.subplots_adjust(left=0.065, right=0.925, bottom=0.045, top=0.960,
                    wspace=0.000, hspace=0.285)
for row, (forest, data) in enumerate(SITE_DATA.items()):
    plot_split_and_gedi_pair(forest, data, axes[row], letters=None)
    if forest != "Agadir":
        axes[row, 0].set_xlabel("")

# Draw first, then centre each modest title over the full histogram-map row.
fig.canvas.draw()
for row, (forest, data) in enumerate(SITE_DATA.items()):
    left = axes[row, 0].get_position()
    right = axes[row, 1].get_position()
    fig.text(
        (left.x0 + right.x1) / 2,
        max(left.y1, right.y1) + 0.014,
        {
            "Ifran": "Ifran — Moderately dense Atlas cedar forest",
            "Maamoura": "Maamoura — Low-density cork-oak woodland",
            "Agadir": "Agadir — Sparse argan woodland",
        }[forest],
        ha="center", va="bottom", fontsize=13.0, fontweight="semibold",
    )

split_gedi_combined_paths = save_figure(
    fig, "Fig_Study_Area_Spatial_Split_and_GEDI_Bins_Three_Forests"
)
# Canonical article asset: a single PDF avoids six-panel LaTeX rescaling artefacts.
article_fullwidth_path = ARTICLE_IMAGES_DIR / "Study_Area_Spatial_Splits_ARTICLE.pdf"
fig.savefig(article_fullwidth_path, bbox_inches="tight", pad_inches=0.04, facecolor="white")
plt.show()
plt.close(fig)


# Independent figures are convenient for single-column or ecological-domain layouts.
split_gedi_individual_paths = {}
for forest, data in SITE_DATA.items():
    fig, axes = plt.subplots(
        1, 2,
        figsize=(15.5, 8.5),
        gridspec_kw={"width_ratios": [0.35, 2.00], "wspace": 0.015},
        constrained_layout=True,
    )
    plot_split_and_gedi_pair(forest, data, axes, letters=None)
    paths = save_figure(
        fig, f"Fig_Study_Area_{forest}_Spatial_Split_and_GEDI_Bins"
    )
    split_gedi_individual_paths[forest] = paths
    plt.show()
    plt.close(fig)

print("\nCombined two-column study-area figure:")
print("\n".join(f"  {kind.upper()}: {path}" for kind, path in split_gedi_combined_paths.items()))
print("\nIndependent forest figures:")
for forest, paths in split_gedi_individual_paths.items():
    print(f"  {forest}:")
    print("\n".join(f"    {kind.upper()}: {path}" for kind, path in paths.items()))


## Reproducibility exports

Vector, raster, and final aligned figure exports are written without modifying the source data.

In [ ]:
QGIS_DIR = OUT_DIR / "QGIS_Layers"
QGIS_DIR.mkdir(parents=True, exist_ok=True)

qgis_report = []

for forest, data in SITE_DATA.items():
    gdf = data["patch_gdf"]
    crs = data["profile"]["crs"]

    # --- Shapefile: one file per split ---
    for split in ("train", "val", "test"):
        shp_path = QGIS_DIR / f"{forest}_{split}.shp"
        part = gdf[gdf["split"].eq(split)].copy()
        if part.empty:
            continue
        part.to_file(shp_path, driver="ESRI Shapefile")
        qgis_report.append({
            "forest": forest,
            "type": "shp",
            "layer": split,
            "path": str(shp_path),
            "n_polygons": len(part),
        })

    # --- Shapefile: all splits merged ---
    all_shp = QGIS_DIR / f"{forest}_all_splits.shp"
    gdf.to_file(all_shp, driver="ESRI Shapefile")
    qgis_report.append({
        "forest": forest,
        "type": "shp",
        "layer": "all_splits",
        "path": str(all_shp),
        "n_polygons": len(gdf),
    })

    # --- GeoTIFF: tree-cover density ---
    density_tif = QGIS_DIR / f"{forest}_tree_cover_density.tif"
    density_arr = data.get("density")
    if density_arr is not None and np.isfinite(density_arr).any():
        p = data["profile"]
        profile_tif = {
            "driver": "GTiff",
            "dtype": "float32",
            "width": p["width"],
            "height": p["height"],
            "count": 1,
            "crs": p["crs"],
            "transform": p["transform"],
            "nodata": np.nan,
        }
        with rasterio.open(density_tif, "w", **profile_tif) as dst:
            dst.write(density_arr, 1)
        qgis_report.append({
            "forest": forest,
            "type": "geotiff",
            "layer": "tree_cover_density",
            "path": str(density_tif),
            "n_polygons": "-",
        })

qgis_df = pd.DataFrame(qgis_report)
display(qgis_df)
qgis_df.to_csv(QGIS_DIR / "QGIS_export_manifest.csv", index=False)
print(f"[PASS] QGIS layers exported to {QGIS_DIR}")
for _, row in qgis_df.iterrows():
    print(f"  {row['forest']:10s} {row['type']:10s} {row.get('layer', ''):15s} -> {row['path']}")


In [ ]:
expected = (
    list(split_gedi_combined_paths.values())
    + [
        path
        for forest_paths in split_gedi_individual_paths.values()
        for path in forest_paths.values()
    ]
    + list(combined_paths.values())
    + list(gedi_count_paths.values())
    + list(gedi_percent_paths.values())
    + [
        path
        for collection in (gedi_count_separate_paths, gedi_percent_separate_paths)
        for forest_paths in collection.values()
        for path in forest_paths.values()
    ]
    + [path for site_paths in individual_paths.values() for path in site_paths.values()]
)
missing = [path for path in expected if not path.is_file() or path.stat().st_size == 0]
if missing:
    raise RuntimeError(f"Missing or empty article outputs: {missing}")

audit = pd.DataFrame([
    {"path": str(path), "format": path.suffix.lower().lstrip("."), "size_kib": path.stat().st_size / 1024}
    for path in expected
])
display(audit)
audit.to_csv(OUT_DIR / "Study_Area_output_audit.csv", index=False)
print(f"[PASS] {len(expected)} non-empty publication files verified.")


In [ ]:
# FIGURE2_ARTICLE_ALIGNED_V12
# Figure Q1 RSE : six panneaux independants. La lecture privilegie les
# Correctifs V6 : suppression des n/% ambigus, axe Y partage, lettres de panneaux
# reservee en haut a droite, hors des AOI.

from matplotlib.patches import Patch, Polygon, Rectangle
from shapely.geometry import Polygon as ShapelyPolygon

FIG2_COLORS = {
    "train": "#0072B2",  # Okabe-Ito blue
    "val":   "#E69F00",  # Okabe-Ito orange
    "test":  "#333333",
}
FIG2_LABELS = {"train": "Train", "val": "Val", "test": "Test"}
FIG2_WIDTH_IN = 7.205   # 183 mm double colonne
FIG2_HEIGHT_IN = 5.85


def _clip_patches_to_aoi(data):
    """Patches clippes strictement a l'interieur de l'AOI."""
    aoi = data.get("aoi_geometry")
    gdf = data["patch_gdf"]
    if aoi is None:
        return gdf
    aoi_gdf = gpd.GeoDataFrame(geometry=[aoi], crs=gdf.crs)
    clipped = gpd.clip(gdf, aoi_gdf, keep_geom_type=True)
    # des LineStrings. Ils ne representent pas des patches et apparaissaient
    polygon_mask = clipped.geometry.geom_type.isin(["Polygon", "MultiPolygon"])
    clipped = clipped.loc[polygon_mask & ~clipped.is_empty].copy()
    # Apres decoupage, un patch peut devenir un MultiPolygon compose d'une zone
    def _main_shell(geom):
        if geom.geom_type == "MultiPolygon":
            geom = max(geom.geoms, key=lambda part: part.area)
        return ShapelyPolygon(geom.exterior)

    clipped.geometry = clipped.geometry.apply(_main_shell)
    # Le seuil (5 % de l'aire mediane d'un patch complet) conserve les portions
    full_patch_area = float(gdf.geometry.area.median())
    min_visible_area = 0.05 * full_patch_area
    return clipped.loc[clipped.geometry.area >= min_visible_area].copy()


def _add_map_letter(ax, letter):
    ax.text(
        0.015, 0.985, f"({letter})", transform=ax.transAxes,
        ha="left", va="top", fontsize=10, fontweight="bold", color="black",
        bbox=dict(facecolor="white", edgecolor="none", alpha=0.85, pad=1.4),
        zorder=40,
    )


def _add_hist_letter(ax, letter):
    ax.text(
        0.015, 0.985, f"({letter})", transform=ax.transAxes,
        ha="left", va="top", fontsize=10, fontweight="bold", color="black",
        bbox=dict(facecolor="white", edgecolor="none", alpha=0.88, pad=1.4),
        clip_on=True, zorder=40,
    )


def _add_north_arrow(ax, forest, raster_bounds):
    # afin que la fleche ne touche ni l'AOI ni le label du panneau.
    cx = 0.90 if forest == "Ifran" else 0.945
    top, bottom, half_width = 0.988, 0.906, 0.022
    outer = [(cx, top), (cx - half_width, bottom), (cx + half_width, bottom)]
    inner = [(cx, top), (cx - half_width, bottom), (cx, bottom + 0.013)]
    ax.add_patch(Polygon(
        outer, closed=True, transform=ax.transAxes, clip_on=False,
        facecolor="white", edgecolor="black", linewidth=0.8,
        joinstyle="miter", zorder=30,
    ))
    ax.add_patch(Polygon(
        inner, closed=True, transform=ax.transAxes, clip_on=False,
        facecolor="black", edgecolor="none", zorder=31,
    ))


def _add_fig2_scale_bar(ax, forest, raster_bounds):
    # Barre d'echelle a 2 labels seulement ("0" et la longueur). Sa longueur
    xmin, ymin, xmax, ymax = raster_bounds
    width = xmax - xmin
    height = ymax - ymin
    square_span = max(width, height)
    length = nice_scale_length(width)
    bar_fraction = length / square_span
    x1_ax = 0.945
    x0_ax = x1_ax - bar_fraction
    y_ax = 0.045
    ax.plot([x0_ax, x1_ax], [y_ax, y_ax], transform=ax.transAxes,
            color="white", lw=4.2, solid_capstyle="butt", zorder=31)
    ax.plot([x0_ax, x1_ax], [y_ax, y_ax], transform=ax.transAxes,
            color="black", lw=0.95, solid_capstyle="butt", zorder=32)
    for x_ax in (x0_ax, x1_ax):
        ax.plot([x_ax, x_ax], [y_ax - 0.011, y_ax + 0.011],
                transform=ax.transAxes, color="black", lw=0.75, zorder=32)
    right_label = f"{length/1000:g} km" if length >= 1000 else f"{length:g} m"
    right_text_x = x1_ax + (0.014 if forest == "Ifran" else 0.0)
    for x_ax, lab, ha in ((x0_ax, "0", "center"),
                          (right_text_x, right_label, "center")):
        ax.text(x_ax, y_ax + 0.022, lab, transform=ax.transAxes,
                ha=ha, va="bottom",
                fontsize=6.8, color="black",
                bbox=dict(facecolor="white", edgecolor="none", alpha=0.80, pad=0.4),
                zorder=33)


def _plot_fig2_map(ax, forest, data, letter):
    p = data["profile"]
    raster_extent = (
        p["bounds"].left, p["bounds"].right,
        p["bounds"].bottom, p["bounds"].top,
    )
    raster_bounds = (
        p["bounds"].left, p["bounds"].bottom,
        p["bounds"].right, p["bounds"].top,
    )

    ax.set_facecolor("#FFFDF0")
    ax.add_patch(Rectangle(
        (0, 0), 1, 1, transform=ax.transAxes, facecolor="#FFFDF0",
        edgecolor="none", clip_on=False, zorder=-100,
    ))

    im = ax.imshow(
        data["density"], extent=raster_extent, origin="upper",
        cmap=plt.cm.YlGn, vmin=1, vmax=100,
        aspect="equal", interpolation="nearest", zorder=0,
    )

    # AOI en contour noir epais. On ne trace que les contours exterieurs : les
    if data.get("aoi_geometry") is not None:
        aoi_geom = data["aoi_geometry"]
        if aoi_geom.geom_type == "Polygon":
            aoi_shells = [ShapelyPolygon(aoi_geom.exterior)]
        else:
            aoi_parts = list(aoi_geom.geoms)
            largest_aoi_area = max(part.area for part in aoi_parts)
            aoi_shells = [
                ShapelyPolygon(part.exterior) for part in aoi_parts
                if part.area >= 0.01 * largest_aoi_area
            ]
        gpd.GeoSeries(aoi_shells, crs=p["crs"]).boundary.plot(
            ax=ax, color="black", linewidth=1.5, zorder=10
        )

    clipped = _clip_patches_to_aoi(data)
    for split in ("train", "val", "test"):
        part = clipped[clipped["split"].eq(split)]
        if len(part) == 0:
            continue
        if split == "test":
            part.plot(
                ax=ax, facecolor="#F0F0F0", edgecolor="black",
                hatch="////", linewidth=1.4, alpha=1.0, zorder=20,
            )
        else:
            part.plot(
                ax=ax, facecolor="none", edgecolor=FIG2_COLORS[split],
                linewidth=2.0, alpha=1.0, zorder=20,
            )

    # Les trois cartes occupent exactement la meme boite physique.  Une emprise
    # site (Agadir etait auparavant plus large que Ifran/Maamoura).
    raster_width = raster_bounds[2] - raster_bounds[0]
    raster_height = raster_bounds[3] - raster_bounds[1]
    square_span = max(raster_width, raster_height)
    center_x = 0.5 * (raster_bounds[0] + raster_bounds[2])
    center_y = 0.5 * (raster_bounds[1] + raster_bounds[3])
    ax.set_xlim(center_x - 0.5 * square_span, center_x + 0.5 * square_span)
    ax.set_ylim(center_y - 0.5 * square_span, center_y + 0.5 * square_span)
    ax.set_aspect("equal", adjustable="box")
    ax.set_box_aspect(1.0)
    ax.set_axis_off()
    # set_axis_off masque normalement le patch de l'axe; on le reactive afin
    ax.patch.set_visible(True)
    _add_map_letter(ax, letter)
    _add_north_arrow(ax, forest, raster_bounds)
    _add_fig2_scale_bar(ax, forest, raster_bounds)
    return im


def _plot_fig2_hist(ax, forest, data, letter):
    shots = GEDI_SHOTS[forest]

    # Bins de 5 m jusqu'a la derniere classe non vide du site
    max_rh = float(shots["rh95"].max())
    bin_edges = np.arange(0.0, max_rh + 5.0, 5.0)
    if len(bin_edges) < 2:
        bin_edges = np.array([0.0, 5.0])
    labels = [f"{int(lo)}-{int(hi)} m" for lo, hi in zip(bin_edges[:-1], bin_edges[1:])]

    work = shots.copy()
    work["height_class"] = pd.cut(
        work["rh95"], bins=bin_edges, labels=labels,
        include_lowest=True, right=False,
    )
    table = (
        work.groupby(["split", "height_class"], observed=False)
        .size()
        .unstack(fill_value=0)
        .reindex(columns=labels, fill_value=0)
    )

    x = np.arange(len(labels))
    width = 0.25

    for offset, split in zip((-1, 0, 1), ("train", "val", "test")):
        counts = table.loc[split].to_numpy() if split in table.index else np.zeros(len(labels))
        if split == "test":
            ax.bar(
                x + offset * width, counts, width=width,
                color="#F5F5F5", edgecolor="black", linewidth=0.9,
                hatch="////", label=FIG2_LABELS[split], zorder=3,
            )
        else:
            ax.bar(
                x + offset * width, counts, width=width,
                color=FIG2_COLORS[split], edgecolor="black", linewidth=0.45,
                label=FIG2_LABELS[split], zorder=2,
            )

    # Marge sobre au-dessus de la barre la plus haute.
    global_max = float(table.values.max())
    ax.set_ylim(0, global_max * 1.14)

    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=30, ha="right", fontsize=7.5)
    ax.tick_params(axis="y", labelsize=7.5)
    ax.grid(axis="y", linestyle=":", alpha=0.45, zorder=0)
    ax.set_axisbelow(True)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    _add_hist_letter(ax, letter)


# --- Construction de la figure par panneaux independants ---
fig = plt.figure(figsize=(FIG2_WIDTH_IN, FIG2_HEIGHT_IN))

site_titles = {
    "Ifran": "Ifran\nCedar-dominated forest",
    "Maamoura": "Maamoura\nCork-oak-dominated woodland",
    "Agadir": "Agadir\nArgan-dominated woodland",
}

# Les trois cartes utilisent maintenant des cellules physiques strictement
# identiques. Les differences de forme des AOI restent visibles a l'interieur
map_left, map_right = 0.030, 0.925
map_gap = 0.012
map_y = 0.545
map_w = (map_right - map_left - 2 * map_gap) / 3
map_h = map_w * FIG2_WIDTH_IN / FIG2_HEIGHT_IN

map_images = []
panel_axes = []
map_axes = []
map_positions = []
x_cursor = map_left
for col, (forest, data) in enumerate(SITE_DATA.items()):
    target_position = [x_cursor, map_y, map_w, map_h]
    ax_map = fig.add_axes(target_position)
    im = _plot_fig2_map(ax_map, forest, data, letter=chr(ord("a") + col))
    map_images.append(im)
    panel_axes.append(ax_map)
    map_axes.append(ax_map)
    map_positions.append(target_position)
    x_cursor += map_w + map_gap

# reference unique. Leur position ne depend donc plus de la forme du raster.
title_y = map_y + map_h + 0.018
for ax_map, forest in zip(map_axes, SITE_DATA):
    pos = ax_map.get_position()
    fig.text(
        pos.x0 + pos.width / 2, title_y, site_titles[forest],
        ha="center", va="bottom", fontsize=8.5, fontweight="semibold",
        linespacing=1.05,
    )

# Histogrammes : trois axes reellement larges et independants, separes par un
hist_left, hist_right = 0.045, 0.940
hist_gap = 0.040
# Translation verticale de 0,5 cm de toute la rangee inferieure.
hist_shift_05cm = (0.5 / 2.54) / FIG2_HEIGHT_IN
hist_y, hist_h = 0.115 + hist_shift_05cm, 0.360
hist_w = (hist_right - hist_left - 2 * hist_gap) / 3
for col, (forest, data) in enumerate(SITE_DATA.items()):
    ax_hist = fig.add_axes([
        hist_left + col * (hist_w + hist_gap), hist_y, hist_w, hist_h
    ])
    _plot_fig2_hist(ax_hist, forest, data, letter=chr(ord("d") + col))
    panel_axes.append(ax_hist)

# et libere les marges entre panneaux.
fig.text(
    -0.032, hist_y + hist_h / 2, "GEDI shots", rotation=90,
    ha="center", va="center", fontsize=8.5,
)
fig.text(
    0.485, 0.018 + hist_shift_05cm, "GEDI RH95 height class",
    ha="center", va="center", fontsize=8.5,
)

# Colorbar tree-cover commune, a droite de la rangee de cartes
cax = fig.add_axes([0.945, map_y + 0.035, 0.015, max(0.18, map_h - 0.070)])
cbar = fig.colorbar(map_images[0], cax=cax)
cbar.set_label("Tree probability (%)", fontsize=8)
cbar.set_ticks([1, 25, 50, 75, 100])
cbar.ax.tick_params(labelsize=7.5)

# Legende commune unique en haut
legend_handles = [
    Patch(facecolor="none", edgecolor=FIG2_COLORS["train"], linewidth=2.0,
          label=FIG2_LABELS["train"]),
    Patch(facecolor="none", edgecolor=FIG2_COLORS["val"], linewidth=2.0,
          label=FIG2_LABELS["val"]),
    Patch(facecolor="#F5F5F5", edgecolor="black", hatch="////", linewidth=0.9,
          label=FIG2_LABELS["test"]),
]
fig.legend(
    handles=legend_handles, loc="upper center", ncol=3, frameon=True,
    bbox_to_anchor=(0.5, 1.055), fontsize=8.5, columnspacing=2.2,
)

fig.canvas.draw()
for ax_map, target_position in zip(map_axes, map_positions):
    ax_map.set_position(target_position)
renderer = fig.canvas.get_renderer()
for letter, ax in zip("abcdef", panel_axes):
    bbox = ax.get_tightbbox(renderer).transformed(fig.dpi_scale_trans.inverted())
    bbox = bbox.expanded(1.025, 1.04)
    fig.savefig(
        OUT_DIR / f"fig2_panel_{letter}.pdf", bbox_inches=bbox,
        facecolor="white",
    )
    fig.savefig(
        OUT_DIR / f"fig2_panel_{letter}.png", dpi=EXPORT_DPI,
        bbox_inches=bbox, facecolor="white",
    )

fig2_paths = save_figure(fig, "fig2_study_areas")
plt.show()
plt.close(fig)

print("\n".join(f"Saved {kind.upper()}: {path}" for kind, path in fig2_paths.items()))
